# Policy Learning

架构图如下：

![policy learning](assets/policy_learning_arch.png)

## 1. REINFORCE

REINFORCE 是一个相对简单的方法，其网络结构约等于原来的 DQN 加上一层 Softmax,后续决策时直接根据输出的最大值采取行动。

代码如下

In [1]:
# %load src/policy_learning/reinforce.py
import os

import gymnasium as gym
import torch
import torch.nn as nn
import torch.optim as optim
from torch.distributions import Categorical


# -------------------------
# Environment
# -------------------------

env = gym.make("CartPole-v1")

state_dim = env.observation_space.shape[0]   # 4
action_dim = env.action_space.n              # 2

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"using device: {device}")

# -------------------------
# Policy Network
# -------------------------

policy = nn.Sequential(
    nn.Linear(state_dim, 128),
    nn.ReLU(),
    nn.Linear(128, action_dim)
).to(device)

optimizer = optim.Adam(policy.parameters(), lr=1e-3)

gamma = 0.99


# -------------------------
# Training
# -------------------------

for episode in range(500):

    state, _ = env.reset()

    log_probs = []
    rewards = []

    while True:

        # 1. policy network outputs logits
        state_tensor = torch.tensor(state, dtype=torch.float32, device=device)
        logits = policy(state_tensor)

        # 2. convert logits to categorical policy
        dist = Categorical(logits=logits)

        # 3. sample action from policy
        action = dist.sample()

        # save log π(a|s)
        log_probs.append(dist.log_prob(action))

        # 4. interact with environment
        next_state, reward, terminated, truncated, _ = env.step(action.item())

        rewards.append(reward)

        state = next_state

        if terminated or truncated:
            break

    # -------------------------
    # Compute returns G_t
    # -------------------------

    returns = []

    G = 0
    for reward in reversed(rewards):
        G = reward + gamma * G
        returns.append(G)

    returns.reverse()
    returns = torch.tensor(returns, dtype=torch.float32, device=device)

    # optional normalization
    returns = (returns - returns.mean()) / (returns.std() + 1e-8)

    # -------------------------
    # REINFORCE loss
    #
    # L = - sum G_t log π(a_t | s_t)
    # -------------------------

    loss = 0

    for log_prob, G in zip(log_probs, returns):
        loss += -log_prob * G

    # -------------------------
    # Gradient update
    # -------------------------

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    total_reward = sum(rewards)
    
    if (episode + 1) % 50 == 0:
        print(
            f"episode={episode}, "
            f"reward={total_reward}"
        )

os.makedirs("checkpoints", exist_ok=True)
torch.save(policy.state_dict(), "checkpoints/reinforce_cartpole.pt")
print("model saved to checkpoints/reinforce_cartpole")

env.close()


## 2. Actor-Critic

REINFORCE 很大的一个问题是，我们每次都是跑完完整的一轮之后再更新参数，这样，假如最终梯度为负，就算这个长长的过程中有一些较优的动作，也会因为最终梯度跟着受抑制。Actor-Critic 通过引入一个评判器，可以在每步都进行抉择。

In [2]:
# %load src/policy_learning/actor-critic.py
import os

import gymnasium as gym
import torch
import torch.nn as nn
import torch.optim as optim
from torch.distributions import Categorical

# Environment

env = gym.make("CartPole-v1")

state_dim = env.observation_space.shape[0]   # 4
action_dim = env.action_space.n              # 2

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"using device: {device}")

# --------------------------------
# Actor-Critic Network
# --------------------------------

actor = nn.Sequential(
    nn.Linear(state_dim, 128),
    nn.ReLU(),
    nn.Linear(128, action_dim),
).to(device)

critic = nn.Sequential(
    nn.Linear(state_dim, 128),
    nn.ReLU(),
    nn.Linear(128, 1)
).to(device)

optimizer = optim.Adam(list(actor.parameters()) + list(critic.parameters()), lr=1e-3)
gamma = 0.99


# --------------------------------
# Training
# --------------------------------

for episode in range(500):

    state, _ = env.reset()
    total_reward = 0

    while True:

        state_tensor = torch.tensor(state, dtype=torch.float32, device=device)

        # --------------------------------
        # Actor + Critic
        # --------------------------------

        logits, value = actor(state_tensor), critic(state_tensor)

        dist = Categorical(logits=logits)
        action = dist.sample()
        log_prob = dist.log_prob(action)

        # --------------------------------
        # Environment step
        # --------------------------------

        next_state, reward, terminated, truncated, _ = env.step(action.item())

        done = terminated or truncated

        next_state_tensor = torch.tensor(next_state, dtype=torch.float32, device=device)

        # --------------------------------
        # TD target
        #
        # y = r + gamma V(s')
        # --------------------------------

        with torch.no_grad():

            next_value = critic(next_state_tensor)

            if terminated:
                target = torch.tensor(reward, dtype=torch.float32, device=device)
            else:
                target = reward + gamma * next_value.squeeze()

        value = value.squeeze()

        # --------------------------------
        # TD error / Advantage
        # --------------------------------

        advantage = target - value

        actor_loss = -log_prob * advantage.detach()
        critic_loss = advantage ** 2
        loss = actor_loss + critic_loss

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        state = next_state
        total_reward += reward

        if done:
            break
    
    if (episode + 1) % 50 == 0:
        print(
            f"episode={episode}, "
            f"reward={total_reward}"
        )

os.makedirs("checkpoints", exist_ok=True)
torch.save(actor.state_dict(), "checkpoints/actor_critic_cartpole.pt")
print("saved to checkpoints/actor_critic_cartpole.pt")

env.close()


二者共用一个 `demo.py`作为可视化演示。

In [3]:
# %load src/policy_learning/demo.py
import argparse

import gymnasium as gym
import torch
import torch.nn as nn

# Parse Input

parser = argparse.ArgumentParser()
parser.add_argument("name", help="model name(REINFORCE/AC)")
args = parser.parse_args()

model_path = None
if args.name == "REINFORCE":
    model_path = "checkpoints/reinforce_cartpole.pt"
elif args.name == "AC":
    model_path = "checkpoints/actor_critic_cartpole.pt"
else:
    raise RuntimeError("model name dosn't exist!")

# Preparing Environment

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"using device: {device}")

env = gym.make("CartPole-v1", render_mode="human")

state_dim = env.observation_space.shape[0]
action_dim = env.action_space.n

policy = nn.Sequential(
    nn.Linear(state_dim, 128),
    nn.ReLU(),
    nn.Linear(128, action_dim),
).to(device)
policy.load_state_dict(torch.load(model_path, map_location=device))
policy.eval()

for episode in range(5):
    state, _ = env.reset()
    total_reward = 0

    while True:
        state_tensor = torch.tensor(state, dtype=torch.float32, device=device)
        with torch.no_grad():
            logits = policy(state_tensor)
            action = logits.argmax().item()

        state, reward, terminated, truncated, _ = env.step(action)
        total_reward += reward
        if terminated or truncated:
            break

    print(
        f"episode={episode}, "
        f"reward={total_reward}"
    )

env.close()

